### Introduzione al Web Caching
Il **web caching** è la tecnica utilizzata per salvare copie di risorse web (come pagine HTML, immagini, video) su server intermedi detti cache server, di modo da servire più velocemente le richieste degli utenti e ridurre il carico sui server originali.

Il problema principale che il web caching cerca di risolvere è il **data delivery bottleneck**: quando molti utenti richiedono le stesse risorse, questo può portare a latenza, congestione di banda e sovraccarico del server originale. 

Il meccanismo base è il seguente: 
- l'utente richiede una risorsa web
- la cache controlla se ne possiede una copia aggiornata, se sì la restituisce subito (**cache hit**), altrimenti la scarica dal server originale, la memorizza e la restituisce all'utente (**cache miss**).

In questo modo, se in futuro un altro utente dovesse richiedere la stessa risorsa al cache server, questa potrà essere servita immediatamente.

I cache server seguono inoltre una gerarchia: cache nel browser (locale nel dispositivo dell'utente), cache proxy dell'ISP, e infine le **CDN (Content Delivery Network)** che sono vere e proprie reti di cache distribuite geograficamente (come quelle di Akamai o Cloudflare).

Guardando il tutto da un punto di vista distribuito, possiamo vedere il **Cache System** come un insieme di **nodi cache** server che devono essere disposti nella rete in modo che ogni **nodo user** abbia qualche nodo cache vicino a cui rivolgersi per le richieste. Si vuole organizzare la rete in modo che questa gestisca nel modo più efficiente possibile le richieste.

<img src="img/cache_system.png" alt="Cache System" width="300">

"Gestire nel modo più efficiente possibile" significa potenzialmente cercare di affrontare molti problemi:
- gestire correttamente un sistema enorme e distribuito, senza un controllo centrale unico
- nodi diversi possono avere informazioni incoerenti tra loro 
- il sistema deve scalare bene all'aumentare di utenti e server
- bisogna evitare hotspot (server molto richiesti) sovraccarichi (fenomeno di "swamping")
- si deve ridurre il traffico della rete, mantenendo basso il tempo necessario per rispondere alle query degli utenti
- si vuole garantire load balancing tra i nodi cache

Il punto importante è che **Internet è un sistema dinamico**, quindi molti server che si spengono o che si aggiungono, utenti che vedono insiemi diversi di server, traffico non uniforme etc... come progettare un sistema di caching che funzioni bene in questo contesto? Serve un modo rapido, deterministico, condiviso da tutti per decidere quale cache è responsabile di ogni item --> la soluzione è usare **funzioni di hashing**.

### Hashing per il Web Caching
Nel modello CSM Cache System Model introduciamo tre insiemi:
- $I$: **insieme di item** (risorse web)
- $U$: **insieme di utenti** (client che fanno richieste)
- $S$: **insieme di server cache** (nodi che memorizzano le risorse)

In CSM si fa la forte assunzione che **gli itam siano richiesti tutti con la stessa probabilità**. L'obiettivo è distribuire uniformemente il carico delle richieste tra i server cache --> la soluzione è definire una funzione hash $h: I \rightarrow S$ che mappa ogni item a un server cache.

Se si ha $f(i) = b$, allora diremo che il server $b$ è responsabile dell'item $i$. Il funzionamento operativo è il seguente:
- quando l'utente vuole richiedere un item $i$, calcola $h(i)$ per determinare quale server cache è responsabile di quell'item
- L'utente quindi contatta $h(i)$, se il server ha già una copia dell'item la restituisce subito, altrimenti lo scarica dal server originale, lo memorizza e lo restituisce all'utente. Le richieste successive per lo stesso item saranno gestite direttamente dalla cache, senza coinvolgere di nuovo il server originale.

Il punto concettuale fondamentale perché questo meccanismo funzioni è che **la funzione hash sia condivisa tra tutti gli utenti e i server**.

Usare una funzione hash è la scelta più naturale perché ogni item (che è una stringa URL) viene convertito tramite hashing in una stringa di lunghezza fissa (usando ad esempio SHA-1, MD5, etc..) che può essere interpretata come un numero. Se si ha $n$ server cache, si può prendere il risultato del hash modulo $n$ per ottenere un indice che identifica il server responsabile di quell'item.
$$\text{server}(i) = h(i) \mod n$$
Tra l'altro utilizzando una funzione hash crittografica si ottiene una distribuzione uniforme degli item tra i server, evitando che alcuni server siano sovraccarichi mentre altri sono poco utilizzati.

Il problema di questo schema è che questo dipende direttamente dal numero di server $n$: **per la natura dinamica di Internet, se un server cache viene aggiunto oppure fallisce allora $n$ cambia!** Questo comporta anche un **cambiamento drastico della funzione di assegnazione e quindi della mappatura degli item ai server**.

L'immagine qui sotto di reshuffling mostra proprio questo fenomeno: immaginando di utilizzare una funzione hash che assegna un item all'hash dell'item + 1 modulo $n$, quando si passa da $n = 4$ a $n = 5$ server, la funzione hash passa da
$$h(d) = d + 1 \mod 4$$
a
$$h(d) = d + 1 \mod 5$$
e quasi tutti i documenti vengono assegnati a server diversi (punti neri mod 4, quadrati bianchi mod 5 --> solo un punto resta assegnato allo stesso server dopo l'aggiunta del nuovo server!):

<img src="img/reshuffling.png" alt="Reshuffling" width="400">

Tutto ciò è un problema enorme perché **in Internet non è realistico assumere che tutti gli utenti aggiornino immediatamente la loro funzione hash** --> alcuni utenti potrebbero avere ancora la vecchia hash, nel momento della ricerca di un item potrebbero contattare un server cache che non la possiede --> **cache miss** e quindi dover contattare il server originale, aumentando latenza e traffico. Questo fenomeno ripetuto per molti utenti porta quindi a cache miss storm, origin server overload etc...

### Consistent Hashing
Possiamo in modo non proprio preciso ma intuitivo definire il problema per cui un nodo con hash non aggiornato ha una visione inconsistente della rete come problema di **view inconsistency**. La soluzione per ridurre questo problema è il **consistent hashing**.

L'idea rivoluzionaria è la seguente: **sia i server che gli item vengono mappati (hashati) nello stesso spazio geometrico, un cerchio unitario $[0,1)$**. Dopo aver posizionato tutto sul cerchio, un item viene assegnato **al primo server incontrato muovendosi nel cerchio in senso orario**.

L'immagine mostra questo meccanismo: i pallini vuoti sono gli item, quelli pieni i server, e un documento viene inserito nel primo server successivo in senso orario. 

<img src="img/consistent.png" alt="Consistent Hashing" width="350">

La **differenza cruciale** rispetto ad hash(key) mod n è che **aggiungere un nuovo server non cambia completamente la mappatura degli item ai vari server, ma solo una piccola frazione di item (in particolare, quelli che cadono nell'arco immediatamente precedente al nuovo server). Tutti gli altri item restano assegnati agli stessi server!**

In questo senso definiamo due proprietà importanti per il consistent hashing:
1. **Smoothness**: quella detta qua sopra; quando si aggiunge un nuovo server (o un server viene rimosso), solo una piccola frazione di item viene riassegnata a quel server, mentre la maggior parte degli item rimane assegnata agli stessi server di prima.
2. **Balance**: se la funzione hash distribuisce uniformemente i punti sul cerchio --> anche gli item saranno distribuiti abbastanza uniformemente tra i server, evitando sovraccarichi

Introduciamo ora, informalmente, due concetti particolarmente importanti nei sistemi di caching distribuiti. Sappiamo come detto che nel mondo distribuito è possibile che in un dato momento client diversi possano vedere un insieme di server cache diversi (hanno diverse **views**). Definiamo due concetti:
- **Spread**: Considerando tante possibili view diverse, uno stesso item non dovrebbe essere mandato a tantissimi server diversi, ma solo a un piccolo insieme di server candidati. Intendiamo in pratica dire che se un utente non aggiornato ha view $V_1 = \{A, B, C, D\}$ mentre un utente aggiornato $V_2 = \{A, B, C, D, E\}$, allora assumendo che $E$ sia il nuovo server aggiunto e il server subito dopo in senso orario sia $A$, allora gli unici server per cui $V_1$ e $V_2$ differiscono sono $A$ e $E$ (perché alcuni degli item che stavano su $A$ ora stanno su $E$).
- **Load**: a differenza dello spread riguarda il punto di vista del server, non dell'item. Il load chiede in parole povere: "considerato un certo server e tutte le view possibili in un dato istante, di quanti item può quel server diventare responsabile?" --> se il load è alto, significa che quel server potrebbe diventare un hotspot in alcune view, e quindi essere sovraccarico.

### Consistent Hashing: Formalizzazione